# pixabay-image-seed-biblia-completa.ipynb — Semeadura da Bíblia inteira

Mesma lógica do `pixabay-image-seed.ipynb`, mas roda **os 66 livros de
uma vez** (todos os ~511 eventos de `eventos-biblicos.js`) -- feito pra
deixar rodando sozinho em segundo plano, não pra ficar acompanhando.

Diferenças do notebook por livro:
- Sem escolher livro/capítulo -- sempre roda tudo
- Delay bem mais longo entre buscas (evita qualquer problema de limite
  de requisição da API do Pixabay ao longo de uma sessão tão comprida)
- Roda em pedaços (checkpoint simples em arquivo) -- se a sessão do
  Colab cair no meio, rodar de novo continua de onde parou, não começa
  do zero

⚠️ Isso pode levar bastante tempo (centenas de eventos × delay) -- inicie
e deixe rodando, não precisa acompanhar célula por célula.

Depois de rodar, o fluxo de revisão + alocação é o mesmo: abra a
`image-stock`, apague o que não serve, e rode a célula de ALOCAR (pode
usar a deste notebook ou a do `pixabay-image-seed.ipynb`, são idênticas).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q -U groq gspread "mistralai>=1.2.0"

import shutil, sys, json
from pathlib import Path

from google.colab import drive, auth, userdata
from google.auth import default
import gspread

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (gravado pelo repositorio-sincronizar)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt — rode o repositorio-sincronizar pra criá-lo")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
        _na_vm    = {f.name for f in DESTINO.glob("*.py")}

        _fora_do_drive = sorted(_esperados - _no_drive)
        _nao_copiados  = sorted((_esperados & _no_drive) - _na_vm)

        if _nao_copiados:
            # Estão no Drive mas não vieram: é a listagem preguiçosa do mount.
            # Uma segunda passada, com o mount já quente, costuma resolver.
            print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
            for _n in _nao_copiados:
                shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
            _na_vm = {f.name for f in DESTINO.glob("*.py")}
            _nao_copiados = sorted((_esperados & _no_drive) - _na_vm)

        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        if _nao_copiados:
            print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
            for _n in _nao_copiados:
                print(f"     {_n}")
            raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
        print(f"   ✅ os {len(_esperados)} módulos do manifesto estão na VM")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

from groq import Groq
from mistralai.client import Mistral

GROQ_API_KEY = userdata.get("GROQ_KEY")
MISTRAL_API_KEY = userdata.get("MISTRAL_KEY")
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
mistral_client = Mistral(api_key=MISTRAL_API_KEY) if MISTRAL_API_KEY else None
MODELO_GROQ = "qwen/qwen3.6-27b"
MODELO_MISTRAL = "mistral-small-latest"

CHAVE_API_PIXABAY = userdata.get("PIXABAY_KEY")

print("✅ Setup concluído")
print(f"   Pixabay: {'disponível' if CHAVE_API_PIXABAY else '❌ PIXABAY_KEY não encontrada nos Secrets do Colab'}")
print(f"   Groq:    {'disponível' if groq_client else 'não configurado (GROQ_KEY ausente)'}")
print(f"   Mistral: {'disponível' if mistral_client else 'não configurado (MISTRAL_KEY ausente)'}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝

PASTA_DRIVE_RAIZ = "narrated_video"

# ── Planilha de imagens (mesma da image-stock usada no match/vídeo) ───────
ID_PLANILHA_IMAGENS = "1P2LydKeeoU5MsAPNl1qhno5qsD1q5BbMOeTbblOVU1E"
NOME_ABA_IMAGENS = "image-stock"

# ── Biblioteca de match (planilha independente) ───────────────────────────
# Deixe em branco na primeira vez -- o notebook cria a planilha sozinho e
# IMPRIME o ID; copie pra cá depois pra reusar (senão cria uma nova toda vez).
# Se já rodou o pixabay-image-seed.ipynb antes, cole o MESMO ID daquele --
# é a mesma biblioteca, compartilhada entre os dois notebooks.
ID_PLANILHA_BIBLIOTECA_MATCH = "1i67VxksAkWYx1cZ_QeoesGXsW28hcA0p5IIfhjx8VHE"
NOME_ABA_BIBLIOTECA_MATCH = "biblioteca_match"

# ── Léxico (mesmos arquivos usados no match-scene-verse.ipynb) ────────────
NOME_ARQUIVO_EVENTOS = "eventos-biblicos.json"
PASTA_DADOS_LEXICO = f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/dados_lexico"

# ── Busca -- delay mais longo aqui de propósito (sessão comprida, roda sozinho) ──
QUANTIDADE_POR_EVENTO = 5
DELAY_SEGUNDOS = 5

# ── Checkpoint em arquivo -- se a sessão do Colab cair no meio, rodar de
# novo continua de onde parou (não refaz o que já tinha sido semeado) ──────
NOME_ARQUIVO_CHECKPOINT = "checkpoint_pixabay_seed_biblia_completa.txt"

print("=" * 60)
print("⚙️  CONFIGURAÇÃO -- BÍBLIA INTEIRA")
print("=" * 60)
print(f"   Imagens/evento:  {QUANTIDADE_POR_EVENTO}")
print(f"   Delay:           {DELAY_SEGUNDOS}s")
print("=" * 60)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INICIALIZAR — carrega o léxico + abre as planilhas            ║
# ╚══════════════════════════════════════════════════════════════════╝
from drive_utils import DriveClient
from match_pipeline import carregar_lexico_biblico, abrir_ou_criar_biblioteca_match
from pixabay_seed_pipeline import garantir_aba_eventos_semeados
from pixabay_seed_pipeline import carregar_eventos_semeados, semear_imagens_lote

_drive = DriveClient.get()

# ── léxico (só eventos-biblicos.js -- essa semeadura é sempre por evento) ──
_dest_eventos = Path(NOME_ARQUIVO_EVENTOS)
_drive.download_se_ausente(PASTA_DADOS_LEXICO, NOME_ARQUIVO_EVENTOS, _dest_eventos)
if not _dest_eventos.exists():
    raise FileNotFoundError(f"{NOME_ARQUIVO_EVENTOS} não encontrado em {PASTA_DADOS_LEXICO} (local nem Drive).")
with open(_dest_eventos, encoding="utf-8") as _f:
    eventos_biblicos = json.load(_f)
print(f"✅ Léxico: {len(eventos_biblicos)} eventos (todos os livros)")

# ── planilhas ───────────────────────────────────────────────────────────
_spreadsheet_imagens = gc.open_by_key(ID_PLANILHA_IMAGENS)
sheet_imagens = _spreadsheet_imagens.worksheet(NOME_ABA_IMAGENS)

_spreadsheet_biblioteca, aba_biblioteca_match, _id_biblioteca_usado = abrir_ou_criar_biblioteca_match(gc, ID_PLANILHA_BIBLIOTECA_MATCH, NOME_ABA_BIBLIOTECA_MATCH)
aba_eventos_semeados = garantir_aba_eventos_semeados(_spreadsheet_biblioteca)

print(f"✅ image-stock aberta ({len(sheet_imagens.get_all_records())} linha(s) hoje)")
print(f"✅ biblioteca_match aberta/criada")
print(f"✅ eventos_semeados aberta/criada")

# ── restaura o checkpoint do Drive, se a sessão tiver caído antes ─────────
_checkpoint_path = Path(NOME_ARQUIVO_CHECKPOINT)
_drive.download_se_ausente(PASTA_DADOS_LEXICO, NOME_ARQUIVO_CHECKPOINT, _checkpoint_path)
_eventos_feitos_checkpoint = set()
if _checkpoint_path.exists():
    _eventos_feitos_checkpoint = {l.strip() for l in _checkpoint_path.read_text(encoding="utf-8").splitlines() if l.strip()}
    print(f"✅ Checkpoint restaurado: {len(_eventos_feitos_checkpoint)} evento(s) já processado(s) numa sessão anterior")
else:
    print("ℹ️  Sem checkpoint anterior -- começando do zero")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1️⃣  SEMEAR — Bíblia inteira, com checkpoint incremental          ║
# ╚══════════════════════════════════════════════════════════════════╝
if not CHAVE_API_PIXABAY:
    raise RuntimeError("PIXABAY_KEY não encontrada -- confira os Secrets do Colab (ver célula de Setup).")

_ja_cobertos = carregar_eventos_semeados(aba_eventos_semeados) | _eventos_feitos_checkpoint
_alvo = [e for e in eventos_biblicos if str(e.get("id", "")) not in _ja_cobertos]

print(f"📋 {len(_alvo)} evento(s) sem cobertura ainda de {len(eventos_biblicos)} no total "
      f"({len(_ja_cobertos)} já feitos)")
print(f"⏱️  Tempo estimado: ~{len(_alvo) * DELAY_SEGUNDOS / 60:.0f} min (delay de {DELAY_SEGUNDOS}s por evento)")
print()

def _salvar_progresso(evento_id):
    """Grava no checkpoint local + sobe pro Drive a cada evento -- se a
    sessão do Colab cair, rodar de novo pula direto pro que falta."""
    with open(_checkpoint_path, "a", encoding="utf-8") as f:
        f.write(str(evento_id) + "\n")
    try:
        _drive.upload(_checkpoint_path, PASTA_DADOS_LEXICO)
    except Exception:
        pass  # nao trava a semeadura se o upload falhar -- o checkpoint local ainda serve nesta sessao

if _alvo:
    resumo = semear_imagens_lote(
        _alvo, CHAVE_API_PIXABAY, sheet_imagens,
        quantidade_por_evento=QUANTIDADE_POR_EVENTO, delay_segundos=DELAY_SEGUNDOS,
        groq_client=groq_client, mistral_client=mistral_client,
        modelo_groq=MODELO_GROQ, modelo_mistral=MODELO_MISTRAL,
        aba_eventos_semeados=aba_eventos_semeados,
        callback_pos_evento=_salvar_progresso,
    )
    print(f"\n✅ {sum(resumo.values())} imagem(ns) nova(s) adicionada(s) na image-stock")
    print(f"✅ Checkpoint atualizado: {len(_alvo)} evento(s) processado(s) nesta sessão")
else:
    print("Nada pra semear -- a Bíblia inteira já tem cobertura.")

print("\n👉 Abra a planilha biblioteca_match e use o menu '📖 Revisão por Versículo' →")
print("   'Abrir painel' pra escolher o vencedor de cada versículo (compara texto + candidatos).")


---
## 2️⃣ Depois de revisar a planilha manualmente, rode a célula abaixo:

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2️⃣  ESCOLHER O VENCEDOR (agora é por versículo, não em lote)     ║
# ╚══════════════════════════════════════════════════════════════════╝
# A alocação automática (1 imagem por evento) foi substituída pelo
# painel de revisão -- cada versículo pode ter sua própria imagem,
# mesmo dentro do mesmo evento/título. Depois de rodar a célula 1
# acima (e revisar/apagar candidatos ruins na image-stock direto),
# abra a planilha biblioteca_match no Sheets e use o menu
# '📖 Revisão por Versículo' → 'Abrir painel' para escolher o vencedor
# de cada versículo, olhando o texto (biblia_texto) ao lado dos
# candidatos (image-stock) -- grava direto na biblioteca_match quando
# você clica.
